## Q1

For each format, the exp() argument bound is set by the **exponent bits (dynamic range)**, not the mantissa bits (precision):
- overflow: `x > ln(max_normal)`
- underflow to zero: `x < ln(min_subnormal)`
- (precision loss starts once `x < ln(min_normal)`, where results fall into the subnormal range)

In [1]:
import math, torch

# (name, exponent_bits, mantissa_bits)
formats = [
    ("FP64",     11, 52),
    ("FP32",      8, 23),
    ("TF32",      8, 10),   # NVIDIA 19-bit: 8 exp, 10 mantissa
    ("BFLOAT16",  8,  7),
    ("FP16",      5, 10),
]

print(f"{'format':9} {'e':>2} {'m':>2} | {'ln(max)':>9} {'ln(min_nrm)':>11} {'ln(min_sub)':>11}")
for name, e, m in formats:
    bias = 2**(e-1) - 1
    max_normal    = 2.0**bias * (2 - 2.0**(-m))
    min_normal    = 2.0**(1 - bias)
    min_subnormal = 2.0**(1 - bias - m)
    print(f"{name:9} {e:2d} {m:2d} | {math.log(max_normal):9.2f} "
          f"{math.log(min_normal):11.2f} {math.log(min_subnormal):11.2f}")

# INT8 is not a float: treat it as unsigned ints 1..255 holding exp() output directly
print(f"\nINT8 (uint 1..255): overflow x > ln(255.5) = {math.log(255.5):.2f}, "
      f"underflow x < ln(0.5) = {math.log(0.5):.2f}")

format     e  m |   ln(max) ln(min_nrm) ln(min_sub)
FP64      11 52 |    709.78     -708.40     -744.44
FP32       8 23 |     88.72      -87.34     -103.28
TF32       8 10 |     88.72      -87.34      -94.27
BFLOAT16   8  7 |     88.72      -87.34      -92.19
FP16       5 10 |     11.09       -9.70      -16.64

INT8 (uint 1..255): overflow x > ln(255.5) = 5.54, underflow x < ln(0.5) = -0.69


**Safe exp() argument range (before overflow / before underflow-to-zero):**

| format | exp / mant bits | overflow at x > | underflow-to-0 at x < |
|---|---|---|---|
| FP64     | 11 / 52 | 709.78 | -744.44 |
| FP32     | 8 / 23  | 88.72  | -103.28 |
| TF32     | 8 / 10  | 88.72  | -94.27  |
| BFLOAT16 | 8 / 7   | 88.72  | -92.19  |
| FP16     | 5 / 10  | 11.09  | -16.64  |
| INT8     | uint 1..255 | 5.54 | -0.69 |

**Key point:** FP32, TF32, BFLOAT16 share the same 8 exponent bits, so they overflow/underflow at the *same* argument (~+89 / -87 for the normal range); they differ only in precision (mantissa). This is exactly why BFLOAT16 is preferred over FP16 for training: it keeps FP32's dynamic range and resists overflow, whereas FP16's 5-bit exponent gives a cramped [-9.7, +11.1] window that overflows easily (hence loss-scaling in FP16 mixed precision). INT8 has essentially no usable range for exp — motivating Q2.